# Phase 3 — Reasoning finetuning on Kaggle

Finetunes one pretrained model on its own language's synthetic comparative-reasoning
data, then scores pretrained vs finetuned on the held-out test set. Run **once per
language** (`hindi`, then `assamese`).

## One-time setup

Two private Kaggle Datasets:

1. **`lma-code`** — run `python kaggle/stage_code_payload.py` locally and upload
   `kaggle/code_payload/` (~1.7 MB). If you already have this dataset from Phase 2,
   upload a **new version**: it now also carries `common/reasoning/`.
2. **`lma-pretrained`** — upload `phase3_pretrained/` (~572 MB), which holds
   `hindi/best.pt` and `assamese/best.pt` from Phase 2.

The reasoning data is **not** uploaded. It is regenerated here from a fixed seed in
a few seconds, which is byte-identical to the local copy and avoids a 38 MB upload.
Four splits are produced: `train`, `val`, `test`, and `test_unseen` (entities held
out of training, used as a generalisation probe rather than the headline metric).

Then: *Add Data* → attach both · *Settings → Accelerator → GPU* · *Internet On*
(only needed for `sentencepiece`, if the image lacks it).

Finetuning is short — a few hundred steps on ~30-token examples — so a session
should not be interrupted. It resumes from `last.pt` regardless if one is.

In [ ]:
# ---- Run configuration -------------------------------------------------
LANG        = "hindi"        # "hindi" or "assamese" -- run once per language
EPOCHS      = 3              # 40,000 examples / batch 32 = 1,250 steps per epoch
BATCH_SIZE  = 32
LEARNING_RATE = 5e-5         # a sixth of pretraining's 3e-4: the weights are
                             # already converged, and the full rate would
                             # overwrite general language knowledge before the
                             # task is learned
EVAL_EVERY  = 250
WORK        = "/kaggle/working"

# Set True to wipe this language's finetuned checkpoints before starting.
FRESH_START = True

In [ ]:
# ---- Code, data and dependencies ---------------------------------------
# The repo is private, so the code arrives as a Kaggle Dataset rather than a
# clone. Both mounts are found by content, because Kaggle derives the mount name
# from the dataset title and may keep the folder that was dragged in.
import os, sys, shutil, subprocess, pathlib

SRC = "/kaggle/working/repo"

def find_mount(marker, what):
    hits = sorted(pathlib.Path("/kaggle/input").glob(f"*/**/{marker}"),
                  key=lambda q: len(q.parts))
    if not hits:
        print("Attached datasets:", sorted(os.listdir("/kaggle/input")) or "(none)")
        raise SystemExit(f"Could not find {what} under /kaggle/input.")
    return hits[0]

code_root = find_mount("common/modeling/model.py", "the `lma-code` payload").parents[2]
ckpt_root = find_mount("hindi/best.pt", "the `lma-pretrained` checkpoints").parents[1]
print("code       ", code_root)
print("checkpoints", ckpt_root)

if os.path.exists(SRC):
    shutil.rmtree(SRC)
shutil.copytree(code_root, SRC)
os.chdir(SRC)
sys.path.insert(0, SRC)

# Drop cached common.* so a re-run picks up a newly uploaded dataset version
# instead of the module object from the previous run.
_stale = [m for m in sys.modules if m == "common" or m.startswith("common.")]
for _m in _stale:
    del sys.modules[_m]
if _stale:
    print(f"dropped {len(_stale)} cached common.* modules")

_ver = pathlib.Path(SRC) / "PAYLOAD_VERSION.txt"
print("payload version:", _ver.read_text().strip().replace("\n", " | ")
      if _ver.exists() else "UNSTAMPED")

import importlib.util
if importlib.util.find_spec("sentencepiece") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentencepiece"])
    importlib.invalidate_caches()

import torch
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    cap = torch.cuda.get_device_capability()
    print(f"GPU: {p.name}  {p.total_memory/1e9:.1f} GB  sm_{cap[0]}{cap[1]}")
    print("training will use:",
          "bfloat16" if cap[0] >= 8 else "float16 (+GradScaler)")
else:
    print("WARNING: no GPU -- Settings > Accelerator > GPU")

PRETRAINED = str(ckpt_root / LANG / "best.pt")
print("pretrained checkpoint:", PRETRAINED)

In [ ]:
# ---- Regenerate the reasoning data -------------------------------------
# Deterministic from a fixed seed, so this reproduces the local dataset exactly
# rather than needing it uploaded. Takes a few seconds.
import subprocess, sys, json, pathlib

subprocess.run([sys.executable, "-m", "common.reasoning.generate",
                "--lang", LANG], check=True)

for split in ("train", "val", "test", "test_unseen"):
    p = pathlib.Path(f"{LANG}/train/reasoning_{split}.jsonl")
    n = sum(1 for _ in p.open(encoding="utf-8"))
    print(f"  {split:12s} {n:6,} examples  {p}")

stats = json.loads(pathlib.Path(f"{LANG}/stats/reasoning_dataset.json")
                   .read_text(encoding="utf-8"))
print("\n  leakage controls:", json.dumps(stats["leakage_controls"], indent=2,
                                          ensure_ascii=False)[:400])
print("\n  a sample prompt:")
row = json.loads(pathlib.Path(f"{LANG}/train/reasoning_test.jsonl")
                 .open(encoding="utf-8").readline())
print("   ", row["prompt"])
print("    answer:", row["answer"])

In [ ]:
# ---- Score the PRETRAINED model first ----------------------------------
# This is the "before" number the report compares against, and it has to be
# measured before any weight is touched.
import subprocess, sys, pathlib

pathlib.Path(f"{WORK}/{LANG}/eval").mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, "-m", "common.reasoning.evaluate",
                "--lang", LANG, "--pretrained", PRETRAINED,
                "--split", "test",
                "--out", f"{WORK}/{LANG}/eval/reasoning_pretrained.json"],
               check=True)

In [ ]:
# ---- Clear previous finetuning output, if asked ------------------------
import shutil, pathlib

out_dir = pathlib.Path(WORK) / LANG / "finetuned"
if FRESH_START and out_dir.exists():
    shutil.rmtree(out_dir)
    print(f"FRESH_START: cleared {out_dir}")
out_dir.mkdir(parents=True, exist_ok=True)
print("output ->", out_dir)

In [ ]:
# ---- Finetune ----------------------------------------------------------
# Starts from the pretrained weights only: the optimizer state in best.pt
# describes the end of a 6,000-step cosine decay and would fight a fresh warmup.
import json
from dataclasses import asdict
from common.reasoning.finetune import FinetuneConfig, Finetuner

cfg = FinetuneConfig(
    language=LANG, pretrained=PRETRAINED, out_dir=str(out_dir),
    epochs=EPOCHS, batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE,
    eval_every=EVAL_EVERY, checkpoint_every=EVAL_EVERY,
)
ft = Finetuner(cfg)
(out_dir / "config.json").write_text(
    json.dumps({"model": asdict(ft.mcfg), "finetune": asdict(cfg)}, indent=2))
print("hyperparameters written to config.json\n")

ft.maybe_resume()
summary = ft.train()

In [ ]:
# ---- Finetuning curves -------------------------------------------------
import json, pathlib
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

recs = [json.loads(l) for l in (out_dir / "finetune_log.jsonl").read_text().splitlines()]
tr = [(r["step"], r["train_loss"]) for r in recs if "train_loss" in r]
va = [(r["step"], r["val_loss"])   for r in recs if "val_loss"   in r]
em = [(r["step"], r["exact_match"]) for r in recs if "exact_match" in r]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
ax1.plot(*zip(*tr), label="Training loss", linewidth=1.0, alpha=0.85)
if va:
    ax1.plot(*zip(*va), label="Validation loss", marker="o", markersize=4, linewidth=1.5)
ax1.set_title(f"{LANG.title()} — reasoning finetuning loss")
ax1.set_xlabel("Finetuning step")
ax1.set_ylabel("Cross-entropy on answer tokens (nats)")
ax1.legend(); ax1.grid(alpha=0.3)

if em:
    ax2.plot(*zip(*em), label="Exact match (validation)", marker="s",
             markersize=4, linewidth=1.5, color="#B4741E")
    ax2.axhline(0.4556, linestyle="--", linewidth=1, color="grey",
                label="Chance baseline")
ax2.set_title(f"{LANG.title()} — reasoning accuracy during finetuning")
ax2.set_xlabel("Finetuning step")
ax2.set_ylabel("Exact-match accuracy")
ax2.set_ylim(0, 1)
ax2.legend(); ax2.grid(alpha=0.3)

fig.tight_layout()
path = out_dir / f"finetune_curve_{LANG}.png"
fig.savefig(path, dpi=150)
print("saved", path)
plt.show()

In [ ]:
# ---- Score the FINETUNED model, and compare ----------------------------
# `test` is the headline number: entities the model has seen, phrasings it has
# not. `test_unseen` additionally withholds the entities -- a harder probe of
# whether the model selects from the prompt or just learned which tokens are
# plausible answers. Both are reported; neither is hidden.
import subprocess, sys

for split in ("test", "test_unseen"):
    print(f"===== {split} =====")
    subprocess.run([sys.executable, "-m", "common.reasoning.evaluate",
                    "--lang", LANG,
                    "--pretrained", PRETRAINED,
                    "--finetuned", str(out_dir / "best.pt"),
                    "--split", split,
                    "--out", f"{WORK}/{LANG}/eval/reasoning_{split}.json"],
                   check=True)

In [ ]:
# ---- Attention: pretrained vs finetuned --------------------------------
# Spec 3.2 asks for a comparison on comparative-reasoning prompts, so both
# checkpoints are probed on the same reasoning prompt rather than on corpus text.
import subprocess, sys

for tag, ckpt in (("pretrained", PRETRAINED), ("finetuned", str(out_dir / "best.pt"))):
    subprocess.run([sys.executable, "-m", "common.reasoning.attention_compare",
                    "--lang", LANG, "--checkpoint", ckpt, "--tag", tag,
                    "--out-dir", f"{WORK}/{LANG}/eval"], check=True)

## After the run

`/kaggle/working/<lang>/` holds:

| Path | What |
|---|---|
| `finetuned/best.pt` | best exact-match checkpoint — the Phase 3 deliverable |
| `finetuned/last.pt` | latest step, for resuming |
| `finetuned/finetune_log.jsonl` | per-step loss, lr, and validation accuracy |
| `finetuned/finetune_curve_<lang>.png` | loss and accuracy curves |
| `finetuned/config.json` | every finetuning hyperparameter |
| `eval/reasoning_pretrained.json` | the before number, measured first |
| `eval/reasoning_test.json` | before vs after on the headline test set |
| `eval/reasoning_test_unseen.json` | the same on held-out entities (generalisation probe) |
| `eval/attention_*_<lang>.png` | attention on a reasoning prompt, both checkpoints |

**Save Version → Save & Run All (Commit)** so these persist, then download
`/kaggle/working/<lang>/`.

Back in the repo: the small artifacts go into `<lang>/train/finetuned/` and
`<lang>/eval/`, and the two `.pt` files go to Google Drive with links in the
README — the same split Phase 2 used.